# Feature Engineering and Transfomration of data

this Notebook mainly focus on feature transformation using SQLite3db and then check the insights based on user interactions, top user activities etc. 


## Feature Engineering and Data Transformation

**Purpose:** Create new features and transform existing ones to improve model performance

**Workflow:**
1. Load cleaned EDA dataset
2. Create interaction-based features
3. Generate user behavior features
4. Build product performance features
5. Derive temporal insights
6. Calculate statistical features
7. Apply transformations using SQLAlchemy
8. Export final feature-engineered dataset

**Dataset Info:**
- 26,250 records
- 53 columns (after EDA preprocessing)
- Mix of numerical, categorical, and temporal features
- Ready for advanced feature engineering

**Expected Outcome:**
- 70-100+ engineered features
- Optimized data types
- Database-backed transformations
- Ready for model building

In [2]:
import pandas as pd
import numpy as np
from sqlalchemy import create_engine, Column, Integer, Float, String, Date, DateTime, Boolean
from sqlalchemy.ext.declarative import declarative_base
from sqlalchemy.orm import sessionmaker
import sqlite3
from datetime import datetime
import warnings
warnings.filterwarnings('ignore')

# Load the EDA dataset
df = pd.read_csv(r"e:/dm4ml project/data/processed/final_dataset_eda.csv")

In [3]:
df.shape

(26250, 53)

In [4]:
len(df.columns)

53

In [6]:
df.memory_usage(deep=True).sum() / (1024 ** 2)

np.float64(62.48047924041748)

In [7]:
df.dtypes

user_id                        int64
product_id                     int64
interaction_type                 str
user_rating                    int64
timestamp                        str
id                             int64
title                            str
description                      str
category                         str
price                        float64
discountPercentage           float64
product_rating               float64
stock                          int64
tags                             str
brand                            str
sku                              str
weight                         int64
dimensions                       str
warrantyInformation              str
shippingInformation              str
availabilityStatus               str
reviews                          str
returnPolicy                     str
minimumOrderQuantity           int64
meta                             str
images                           str
thumbnail                        str
b

In [8]:
df.head(10)

,user_id,product_id,interaction_type,user_rating,timestamp,id,title,description,category,price,...,weight_normalized,user_rating_normalized,product_rating_normalized,price_standardized,discount_standardized,stock_standardized,weight_standardized,price_category,discount_category,product_rating_category
0,694,10,click,3,2026-04-12 21:17:02.980005,10,gucci bloom eau de,gucci bloom by gucci is a floral and captivati...,fragrances,79.99,...,0.666667,0.50,0.087500,-0.250368,0.852079,1.074345,0.380121,Mid-range,Low,Average
1,2276,6,click,4,2026-04-07 21:17:02.987003,6,calvin klein ck one,ck one by calvin klein is a classic unisex fra...,fragrances,49.99,...,0.666667,0.75,0.766667,-0.303623,-1.455666,-0.792873,0.380121,Budget,Low,Excellent
2,4642,24,purchase,4,2026-03-24 21:17:02.986514,24,fish steak,quality fish steak suitable for grilling bakin...,groceries,14.99,...,0.555556,0.75,0.520833,-0.365753,-1.023656,0.562366,0.065521,Budget,Low,Good
3,3016,28,click,1,2026-03-20 21:17:02.981622,28,ice cream,creamy and delicious ice cream available in va...,groceries,5.49,...,0.000000,0.00,0.358333,-0.382617,-0.200253,-0.853106,-1.507482,Budget,Low,Good
4,8246,27,click,3,2026-03-17 21:17:02.983616,27,honey jar,pure and natural honey in a convenient jar per...,groceries,6.99,...,0.111111,0.50,0.600000,-0.379954,0.853925,-0.642291,-1.192882,Budget,Low,Good
5,997,2,view,1,2026-03-19 21:17:02.981212,2,eyeshadow palette with mirror,the eyeshadow palette with mirror offers a ver...,beauty,19.99,...,0.888889,0.00,0.137500,-0.356877,1.553633,-0.642291,1.009323,Budget,Low,Average
6,3716,10,view,3,2026-03-24 21:17:02.992953,10,gucci bloom eau de,gucci bloom by gucci is a floral and captivati...,fragrances,79.99,...,0.666667,0.50,0.087500,-0.250368,0.852079,1.074345,0.380121,Mid-range,Low,Average
7,1474,29,view,4,2026-04-06 21:17:02.988141,29,juice,refreshing fruit juice packed with vitamins an...,groceries,3.99,...,0.000000,0.75,0.587500,-0.385280,0.421915,-0.160428,-1.507482,Budget,Low,Good
8,4212,15,purchase,3,2026-04-03 21:17:02.980075,15,wooden bathroom sink with mirror,the wooden bathroom sink with mirror is a uniq...,furniture,799.99,...,1.000000,0.50,0.441667,1.027742,-0.179945,-1.455434,1.323923,Luxury,Low,Good
9,6206,7,click,1,2026-03-28 21:17:02.986834,7,chanel coco noir eau de,coco noir by chanel is an elegant and mysterio...,fragrances,129.99,...,0.666667,0.00,0.720833,-0.161610,1.243472,0.080503,0.380121,Mid-range,Low,Excellent


## Interaction-Based Features

**Purpose:** Create features based on user-product interactions

**Features to Create:**

1. **User-Product Interaction Metrics:**
   - user_product_interaction_count: How many times user interacted with product
   - user_interaction_diversity: Number of unique products user interacted with
   - product_user_diversity: Number of unique users who interacted with product

2. **Interaction Type Features:**
   - interaction_type_encoded: Numerical encoding (view=1, cart=2, purchase=3)
   - has_purchased: Binary flag if user purchased product
   - has_carted: Binary flag if user added to cart
   - interaction_sequence: Order of interaction types

3. **Engagement Metrics:**
   - user_engagement_level: Active/Moderate/Passive based on interaction count
   - product_engagement: High/Medium/Low based on popularity
   - interaction_recency: Days since interaction

**Business Logic:**
- Track user journey through different interaction types
- Identify power users and popular products
- Understand engagement patterns

In [9]:
# 1. User-Product Interaction Count
user_product_interactions = df.groupby(['user_id', 'product_id']).size().reset_index(name='user_product_interaction_count')
df = df.merge(user_product_interactions, on=['user_id', 'product_id'], how='left')

# 2. User Interaction Diversity (unique products per user)
user_product_diversity = df.groupby('user_id')['product_id'].nunique().reset_index(name='user_product_diversity')
df = df.merge(user_product_diversity, on='user_id', how='left')

# 3. Product User Diversity (unique users per product)
product_user_diversity = df.groupby('product_id')['user_id'].nunique().reset_index(name='product_user_diversity')
df = df.merge(product_user_diversity, on='product_id', how='left')

# 4. Interaction Type Encoding
interaction_type_mapping = {
    'view': 1,
    'cart': 2,
    'purchase': 3,
    'add': 2,
    'remove': 1,
    'wishlist': 1
}
df['interaction_type_encoded'] = df['interaction_type'].map(interaction_type_mapping).fillna(1).astype(int)

# 5. Binary Flags for Interaction Types
df['has_purchase'] = (df['interaction_type'].str.lower() == 'purchase').astype(int)
df['has_cart'] = (df['interaction_type'].str.lower() == 'cart').astype(int)
df['has_view'] = (df['interaction_type'].str.lower() == 'view').astype(int)

# 6. User Engagement Level
user_interaction_counts = df.groupby('user_id').size().reset_index(name='total_user_interactions')
df = df.merge(user_interaction_counts, on='user_id', how='left')

def categorize_engagement(count):
    if count >= 20:
        return 'High'
    elif count >= 10:
        return 'Medium'
    else:
        return 'Low'

df['user_engagement_level'] = df['total_user_interactions'].apply(categorize_engagement)

# 7. Product Engagement Level
product_interaction_counts = df.groupby('product_id').size().reset_index(name='total_product_interactions')
df = df.merge(product_interaction_counts, on='product_id', how='left')

df['product_engagement_level'] = df['total_product_interactions'].apply(categorize_engagement)

print("\nInteraction-Based Features Created:")
print(f"  ✓ user_product_interaction_count")
print(f"  ✓ user_product_diversity")
print(f"  ✓ product_user_diversity")
print(f"  ✓ interaction_type_encoded")
print(f"  ✓ has_purchase, has_cart, has_view")
print(f"  ✓ user_engagement_level")
print(f"  ✓ product_engagement_level")

print(f"\nNew feature examples:")
print(df[['user_id', 'product_id', 'interaction_type', 'interaction_type_encoded', 
          'has_purchase', 'user_engagement_level', 'product_engagement_level']].head(10))


Interaction-Based Features Created:
  ✓ user_product_interaction_count
  ✓ user_product_diversity
  ✓ product_user_diversity
  ✓ interaction_type_encoded
  ✓ has_purchase, has_cart, has_view
  ✓ user_engagement_level
  ✓ product_engagement_level

New feature examples:
   user_id  product_id interaction_type  interaction_type_encoded  \
0      694          10            click                         1   
1     2276           6            click                         1   
2     4642          24         purchase                         3   
3     3016          28            click                         1   
4     8246          27            click                         1   
5      997           2             view                         1   
6     3716          10             view                         1   
7     1474          29             view                         1   
8     4212          15         purchase                         3   
9     6206           7            click 

## User Behavior Features

**Purpose:** Capture user patterns and preferences

**Features to Create:**

1. **User Statistical Metrics:**
   - user_avg_rating: Average rating given by user
   - user_rating_std: Standard deviation of user ratings
   - user_avg_price: Average price of products user interacts with
   - user_price_std: Standard deviation of prices

2. **User Preferences:**
   - user_preferred_category: Most frequently interacted category
   - user_preferred_price_range: Budget/Mid-range/Premium based on average
   - user_rating_behavior: Harsh/Normal/Generous based on distribution

3. **User Loyalty:**
   - user_repeat_purchase_rate: Percentage of repeat purchases
   - user_category_concentration: How focused on few categories (0-1)
   - user_brand_loyalty: Interaction diversity across brands

**Insights:**
- Understand user preferences for personalization
- Identify user segments (budget-conscious, quality-seekers, etc.)
- Predict user satisfaction probability

In [10]:
# 1. User Rating Statistics
user_rating_stats = df.groupby('user_id')['user_rating'].agg(['mean', 'std']).reset_index()
user_rating_stats.columns = ['user_id', 'user_avg_rating', 'user_rating_std']
user_rating_stats['user_rating_std'].fillna(0, inplace=True)
df = df.merge(user_rating_stats, on='user_id', how='left')

# 2. User Price Statistics
user_price_stats = df.groupby('user_id')['price'].agg(['mean', 'std']).reset_index()
user_price_stats.columns = ['user_id', 'user_avg_price', 'user_price_std']
user_price_stats['user_price_std'].fillna(0, inplace=True)
df = df.merge(user_price_stats, on='user_id', how='left')

# 3. User Preferred Category
user_preferred_category = df.groupby('user_id')['category'].agg(lambda x: x.value_counts().index[0]).reset_index()
user_preferred_category.columns = ['user_id', 'user_preferred_category']
df = df.merge(user_preferred_category, on='user_id', how='left')

# 4. User Preferred Price Range
def categorize_price_preference(price):
    if price < 100:
        return 'Budget'
    elif price < 500:
        return 'Mid-range'
    else:
        return 'Premium'

df['user_preferred_price_range'] = df['user_avg_price'].apply(categorize_price_preference)

# 5. User Repeat Purchase Rate
user_purchase_stats = df[df['interaction_type'] == 'purchase'].groupby('user_id').size().reset_index(name='user_purchase_count')
df = df.merge(user_purchase_stats, on='user_id', how='left', suffixes=('', '_purchases'))
df['user_purchase_count'].fillna(0, inplace=True)
df['user_repeat_purchase_rate'] = (df['user_purchase_count'] / df['total_user_interactions'] * 100).fillna(0)

# 6. User Category Concentration (Herfindahl Index)
def calculate_concentration(user_id):
    user_categories = df[df['user_id'] == user_id]['category'].value_counts()
    total = user_categories.sum()
    concentration = ((user_categories / total) ** 2).sum()
    return concentration

user_ids = df['user_id'].unique()
concentration_dict = {user_id: calculate_concentration(user_id) for user_id in user_ids}
df['user_category_concentration'] = df['user_id'].map(concentration_dict)

# 7. User Brand Loyalty
user_brand_diversity = df.groupby('user_id')['brand'].nunique().reset_index(name='user_brand_diversity')
df = df.merge(user_brand_diversity, on='user_id', how='left')
df['user_brand_diversity'].fillna(0, inplace=True)

print("\nUser Behavior Features Created:")
print(f"  ✓ user_avg_rating, user_rating_std")
print(f"  ✓ user_avg_price, user_price_std")
print(f"  ✓ user_preferred_category")
print(f"  ✓ user_preferred_price_range")
print(f"  ✓ user_repeat_purchase_rate")
print(f"  ✓ user_category_concentration")
print(f"  ✓ user_brand_diversity")

print(f"\nUser behavior examples:")
print(df[['user_id', 'user_avg_rating', 'user_avg_price', 'user_preferred_category', 
          'user_repeat_purchase_rate', 'user_category_concentration']].drop_duplicates('user_id').head(10))


User Behavior Features Created:
  ✓ user_avg_rating, user_rating_std
  ✓ user_avg_price, user_price_std
  ✓ user_preferred_category
  ✓ user_preferred_price_range
  ✓ user_repeat_purchase_rate
  ✓ user_category_concentration
  ✓ user_brand_diversity

User behavior examples:
   user_id  user_avg_rating  user_avg_price user_preferred_category  \
0      694         3.000000       79.990000              fragrances   
1     2276         2.866667       36.390000              fragrances   
2     4642         4.000000       12.490000               groceries   
3     3016         2.714286        3.690000               groceries   
4     8246         3.888889       12.767778               groceries   
5      997         2.333333      844.767778                  beauty   
6     3716         4.250000       25.990000               groceries   
7     1474         4.000000        3.990000               groceries   
8     4212         3.000000      799.990000               furniture   
9     6206    

## Product Performance Features

**Purpose:** Capture product characteristics and market performance

**Features to Create:**

1. **Product Quality Metrics:**
   - product_avg_rating_gap: Difference between product rating and user average
   - product_rating_tier: Tier based on product rating (Excellent/Good/Average/Poor)
   - product_rating_consistency: Standard deviation of ratings (lower = consistent)

2. **Product Popularity:**
   - product_popularity_score: Normalized interaction count
   - product_sales_velocity: Purchase count / total interactions
   - product_view_to_purchase_ratio: Views vs purchases

3. **Product Economics:**
   - product_discount_frequency: Percentage of interactions with discount
   - product_discounted_price: Price after discount
   - product_margin_impact: Price vs average category price

4. **Product Availability:**
   - product_stock_level: Low/Medium/High/Unavailable
   - days_until_stockout: Estimated days based on velocity

**Insights:**
- Identify high-value vs low-performing products
- Understand pricing strategy effectiveness
- Predict product lifecycle stage

In [11]:
# 1. Product Quality Metrics
product_rating_stats = df.groupby('product_id')['product_rating'].agg(['mean', 'std']).reset_index()
product_rating_stats.columns = ['product_id', 'product_avg_rating', 'product_rating_std']
product_rating_stats['product_rating_std'].fillna(0, inplace=True)
df = df.merge(product_rating_stats, on='product_id', how='left')

# 2. Product Rating Gap (difference from user's average rating)
df['product_rating_gap'] = df['product_rating'] - df['user_avg_rating']

# 3. Product Rating Tier
def categorize_rating_tier(rating):
    if rating >= 4.5:
        return 'Excellent'
    elif rating >= 3.5:
        return 'Good'
    elif rating >= 2.5:
        return 'Average'
    else:
        return 'Poor'

df['product_rating_tier'] = df['product_rating'].apply(categorize_rating_tier)

# 4. Product Sales Velocity
product_purchase_count = df[df['interaction_type'] == 'purchase'].groupby('product_id').size().reset_index(name='product_purchase_count')
df = df.merge(product_purchase_count, on='product_id', how='left')
df['product_purchase_count'].fillna(0, inplace=True)
df['product_sales_velocity'] = (df['product_purchase_count'] / df['total_product_interactions'] * 100).fillna(0)

# 5. Product View to Purchase Ratio
product_view_count = df[df['interaction_type'] == 'view'].groupby('product_id').size().reset_index(name='product_view_count')
df = df.merge(product_view_count, on='product_id', how='left')
df['product_view_count'].fillna(0, inplace=True)
df['product_view_to_purchase_ratio'] = (df['product_purchase_count'] / (df['product_view_count'] + 1)).fillna(0)

# 6. Product Discount Frequency
product_discount_interactions = df[df['discountPercentage'] > 0].groupby('product_id').size().reset_index(name='product_discounted_interactions')
df = df.merge(product_discount_interactions, on='product_id', how='left')
df['product_discounted_interactions'].fillna(0, inplace=True)
df['product_discount_frequency'] = (df['product_discounted_interactions'] / df['total_product_interactions'] * 100).fillna(0)

# 7. Product Discounted Price
df['product_discounted_price'] = df['price'] * (1 - df['discountPercentage'] / 100)

# 8. Product Stock Level
def categorize_stock_level(stock):
    if stock == 0:
        return 'Unavailable'
    elif stock < 20:
        return 'Low'
    elif stock < 100:
        return 'Medium'
    else:
        return 'High'

df['product_stock_level'] = df['stock'].apply(categorize_stock_level)

print("\nProduct Performance Features Created:")
print(f"  ✓ product_avg_rating, product_rating_std")
print(f"  ✓ product_rating_gap")
print(f"  ✓ product_rating_tier")
print(f"  ✓ product_sales_velocity")
print(f"  ✓ product_view_to_purchase_ratio")
print(f"  ✓ product_discount_frequency")
print(f"  ✓ product_discounted_price")
print(f"  ✓ product_stock_level")

print(f"\nProduct performance examples:")
print(df[['product_id', 'product_avg_rating', 'product_rating_tier', 
          'product_sales_velocity', 'product_view_to_purchase_ratio', 
          'product_discount_frequency']].drop_duplicates('product_id').head(10))


Product Performance Features Created:
  ✓ product_avg_rating, product_rating_std
  ✓ product_rating_gap
  ✓ product_rating_tier
  ✓ product_sales_velocity
  ✓ product_view_to_purchase_ratio
  ✓ product_discount_frequency
  ✓ product_discounted_price
  ✓ product_stock_level

Product performance examples:
    product_id  product_avg_rating product_rating_tier  \
0           10                2.74             Average   
1            6                4.37                Good   
2           24                3.78                Good   
3           28                3.39             Average   
4           27                3.97                Good   
5            2                2.86             Average   
7           29                3.94                Good   
8           15                3.59                Good   
9            7                4.26                Good   
10           1                2.56             Average   

    product_sales_velocity  product_view_to_purchase_ra

## Category-Based Features

**Purpose:** Create category-level aggregations and comparisons

**Features to Create:**

1. **Category Market Metrics:**
   - category_avg_price: Average price in category
   - category_avg_rating: Average rating of products in category
   - category_price_variance: Price variation within category
   - category_market_share: Product share of interactions in category

2. **Category Performance:**
   - category_popularity: Total interactions in category
   - category_conversion_rate: Purchase rate in category
   - category_avg_discount: Average discount in category

3. **Product vs Category Comparison:**
   - product_price_vs_category: Price difference from category average
   - product_rating_vs_category: Rating difference from category average
   - product_category_rank: Rank of product within category

**Insights:**
- Identify category trends
- Benchmark products against category standards
- Understand market dynamics

In [12]:
# 1. Category Market Metrics
category_metrics = df.groupby('category').agg({
    'price': ['mean', 'std'],
    'product_rating': 'mean',
    'discountPercentage': 'mean',
    'product_id': 'count'
}).reset_index()

category_metrics.columns = ['category', 'category_avg_price', 'category_price_std', 
                            'category_avg_rating', 'category_avg_discount', 'category_interaction_count']
category_metrics['category_price_std'].fillna(0, inplace=True)

df = df.merge(category_metrics, on='category', how='left')

# 2. Category Conversion Rate
category_purchases = df[df['interaction_type'] == 'purchase'].groupby('category').size().reset_index(name='category_purchase_count')
df = df.merge(category_purchases, on='category', how='left')
df['category_purchase_count'].fillna(0, inplace=True)
df['category_conversion_rate'] = (df['category_purchase_count'] / df['category_interaction_count'] * 100).fillna(0)

# 3. Product vs Category Price
df['product_price_vs_category'] = df['price'] - df['category_avg_price']
df['price_premium_ratio'] = (df['price'] / df['category_avg_price']).fillna(1)

# 4. Product vs Category Rating
df['product_rating_vs_category'] = df['product_rating'] - df['category_avg_rating']

# 5. Product Category Rank (rank by price within category)
df['product_category_price_rank'] = df.groupby('category')['price'].rank()
df['product_category_rating_rank'] = df.groupby('category')['product_rating'].rank(ascending=False)

print("\nCategory-Based Features Created:")
print(f"  ✓ category_avg_price, category_price_std")
print(f"  ✓ category_avg_rating")
print(f"  ✓ category_avg_discount")
print(f"  ✓ category_interaction_count")
print(f"  ✓ category_conversion_rate")
print(f"  ✓ product_price_vs_category")
print(f"  ✓ price_premium_ratio")
print(f"  ✓ product_rating_vs_category")
print(f"  ✓ product_category_price_rank")

print(f"\nCategory-based examples:")
print(df[['category', 'category_avg_price', 'category_avg_rating', 
          'product_price_vs_category', 'product_rating_vs_category', 
          'category_conversion_rate']].drop_duplicates('category').head(10))


Category-Based Features Created:
  ✓ category_avg_price, category_price_std
  ✓ category_avg_rating
  ✓ category_avg_discount
  ✓ category_interaction_count
  ✓ category_conversion_rate
  ✓ product_price_vs_category
  ✓ price_premium_ratio
  ✓ product_rating_vs_category
  ✓ product_category_price_rank

Category-based examples:
     category  category_avg_price  category_avg_rating  \
0  fragrances           83.061688             3.814730   
2   groceries            6.040420             3.849992   
5      beauty           13.396489             3.824155   
8   furniture         1197.300734             4.024432   

   product_price_vs_category  product_rating_vs_category  \
0                  -3.071688                   -1.074730   
2                   8.949580                   -0.069992   
5                   6.593511                   -0.964155   
8                -397.310734                   -0.434432   

   category_conversion_rate  
0                 32.531760  
2                 

## Derived Statistical Features

**Purpose:** Create ratio and interaction features for modeling

**Features to Create:**

1. **Price-Rating Relationships:**
   - price_quality_ratio: Rating per dollar (value proposition)
   - discount_effectiveness: Purchase rate with vs without discount
   - price_discount_interaction: Price * discount percentage

2. **Engagement Ratios:**
   - cart_to_view_ratio: Shopping engagement
   - purchase_to_cart_ratio: Conversion efficiency
   - user_product_interaction_rate: Frequency of interactions

3. **Feature Interactions:**
   - high_rating_budget_product: Boolean (rating > 4 AND price < 100)
   - discount_vulnerable_product: Boolean (high discount rate)
   - user_category_match: User preference alignment

**Benefits:**
- Capture non-linear relationships
- Create meaningful business logic features
- Improve model performance

In [13]:
# 1. Price-Quality Ratio
df['price_quality_ratio'] = df['product_rating'] / (df['price'] + 1)  # +1 to avoid division by zero

# 2. Discount Effectiveness
product_discount_purchase_rate = df[df['discountPercentage'] > 0].groupby('product_id')['has_purchase'].mean().reset_index(name='discount_purchase_rate')
df = df.merge(product_discount_purchase_rate, on='product_id', how='left')
df['discount_purchase_rate'].fillna(0, inplace=True)

product_no_discount_purchase_rate = df[df['discountPercentage'] == 0].groupby('product_id')['has_purchase'].mean().reset_index(name='no_discount_purchase_rate')
df = df.merge(product_no_discount_purchase_rate, on='product_id', how='left')
df['no_discount_purchase_rate'].fillna(0, inplace=True)

df['discount_effectiveness'] = (df['discount_purchase_rate'] - df['no_discount_purchase_rate']).fillna(0)

# 3. Price-Discount Interaction
df['price_discount_interaction'] = df['price'] * (df['discountPercentage'] / 100)

# 4. Cart to View Ratio
product_cart_count = df[df['interaction_type'] == 'cart'].groupby('product_id').size().reset_index(name='product_cart_count')
df = df.merge(product_cart_count, on='product_id', how='left')
df['product_cart_count'].fillna(0, inplace=True)
df['cart_to_view_ratio'] = (df['product_cart_count'] / (df['product_view_count'] + 1)).fillna(0)

# 5. Purchase to Cart Ratio
df['purchase_to_cart_ratio'] = (df['product_purchase_count'] / (df['product_cart_count'] + 1)).fillna(0)

# 6. User-Product Interaction Rate
df['user_product_interaction_rate'] = df['user_product_interaction_count'] / (df['total_user_interactions'] + 1)

# 7. High Rating Budget Product
df['high_rating_budget_product'] = ((df['product_rating'] >= 4) & (df['price'] < 100)).astype(int)

# 8. Discount Vulnerable Product
df['discount_vulnerable_product'] = (df['product_discount_frequency'] > 50).astype(int)

# 9. User Category Match
df['user_category_match'] = (df['category'] == df['user_preferred_category']).astype(int)

print("\nDerived Statistical Features Created:")
print(f"  ✓ price_quality_ratio")
print(f"  ✓ discount_effectiveness")
print(f"  ✓ price_discount_interaction")
print(f"  ✓ cart_to_view_ratio")
print(f"  ✓ purchase_to_cart_ratio")
print(f"  ✓ user_product_interaction_rate")
print(f"  ✓ high_rating_budget_product")
print(f"  ✓ discount_vulnerable_product")
print(f"  ✓ user_category_match")

print(f"\nDerived feature examples:")
print(df[['price_quality_ratio', 'discount_effectiveness', 'cart_to_view_ratio', 
          'purchase_to_cart_ratio', 'high_rating_budget_product']].head(10))


Derived Statistical Features Created:
  ✓ price_quality_ratio
  ✓ discount_effectiveness
  ✓ price_discount_interaction
  ✓ cart_to_view_ratio
  ✓ purchase_to_cart_ratio
  ✓ user_product_interaction_rate
  ✓ high_rating_budget_product
  ✓ discount_vulnerable_product
  ✓ user_category_match

Derived feature examples:
   price_quality_ratio  discount_effectiveness  cart_to_view_ratio  \
0             0.033831                     0.0                 0.0   
1             0.085703                     0.0                 0.0   
2             0.236398                     0.0                 0.0   
3             0.522342                     0.0                 0.0   
4             0.496871                     0.0                 0.0   
5             0.136255                     0.0                 0.0   
6             0.033831                     0.0                 0.0   
7             0.789579                     0.0                 0.0   
8             0.004482                     0.0     

## Database Operations with SQLAlchemy

**Purpose:** Store transformed features in SQLite database for scalability

**Operations:**

1. **Create SQLite Engine:**
   - Initialize connection to SQLite database
   - Define table schemas using SQLAlchemy ORM

2. **Data Persistence:**
   - Store raw features
   - Store engineered features
   - Create indexed tables for fast queries

3. **Aggregations:**
   - User-level aggregations
   - Product-level aggregations
   - Category-level aggregations

**Benefits:**
- Scalable storage beyond memory
- SQL query capabilities
- Data versioning
- Performance optimization with indexing

In [ ]:
import pandas as pd
import gc
from sqlalchemy import create_engine, Column, Integer, String, Float, Boolean
from sqlalchemy.orm import declarative_base

# 1. Setup Engine & Schema
engine = create_engine('sqlite:///feature_engineered_data.db', echo=False)
Base = declarative_base()

class FeatureEngineeringTable(Base):
    __tablename__ = 'engineered_features'
    id = Column(Integer, primary_key=True)
    user_id = Column(Integer, index=True)
    product_id = Column(Integer, index=True)
    interaction_type = Column(String)
    user_rating = Column(Integer)
    product_rating = Column(Float)
    price = Column(Float)
    discount_percentage = Column(Float)
    category = Column(String, index=True)
    stock = Column(Integer)
    brand = Column(String)
    interaction_type_encoded = Column(Integer)
    has_purchase = Column(Boolean)
    user_engagement_level = Column(String)
    product_engagement_level = Column(String)
    price_quality_ratio = Column(Float)
    discount_effectiveness = Column(Float)
    cart_to_view_ratio = Column(Float)
    purchase_to_cart_ratio = Column(Float)

Base.metadata.create_all(engine)
print("✓ Database tables created successfully")

# 2. Downcast numeric columns to save RAM immediately
fcols = df.select_dtypes('float').columns
icols = df.select_dtypes('integer').columns
df[fcols] = df[fcols].apply(pd.to_numeric, downcast='float')
df[icols] = df[icols].apply(pd.to_numeric, downcast='integer')

# 3. Store Main Engineered Features (NO .copy())
print("Storing engineered features...")
columns_to_store = [col for col in df.columns if col in 
    ['user_id', 'product_id', 'interaction_type', 'user_rating', 'product_rating', 
     'price', 'discountPercentage', 'category', 'stock', 'brand'] + 
    [c for c in df.columns if any(x in c for x in ['interaction_type_encoded', 'has_purchase', 'engagement', 'price_quality', 'ratio'])]
]

# Using chunksize=10000 prevents memory spikes during the write process
df[columns_to_store].to_sql(
    'engineered_features', 
    con=engine, 
    if_exists='replace', 
    index=False, 
    chunksize=400, 
    method='multi'
)
print("✓ Engineered features stored to database")


SQLALCHEMY DATABASE OPERATIONS (OPTIMIZED)
✓ Database tables created successfully
Storing engineered features...


✓ Engineered features stored to database


In [16]:
# 4. User-level Aggregations
print("\nCreating user-level aggregations...")
user_aggregations = df.groupby('user_id').agg({
    'product_id': 'nunique',
    'user_rating': ['mean', 'std'],
    'price': ['mean', 'min', 'max'],
    'has_purchase': 'sum',
    'has_cart': 'sum',
    'user_product_diversity': 'first',
    'user_avg_rating': 'first',
    'user_engagement_level': 'first',
    'user_repeat_purchase_rate': 'first'
}).reset_index()

user_aggregations.columns = ['user_id', 'products_interacted', 'avg_rating', 'rating_std',
                             'avg_price', 'min_price', 'max_price', 'purchase_count',
                             'cart_count', 'product_diversity', 'user_avg_rating',
                             'engagement_level', 'repeat_purchase_rate']

user_aggregations.to_sql('user_aggregations', con=engine, if_exists='replace', index=False, chunksize=10000)
del user_aggregations
gc.collect()
print("✓ User aggregations stored")


Creating user-level aggregations...
✓ User aggregations stored


## SQL-Based User Feature Engineering

**Purpose:** Create user-level aggregation features using SQL window functions

**Queries Include:**
1. User interaction history with row numbers
2. User cumulative rating trends
3. User behavior patterns over time
4. User frequency and recency metrics

**SQL Advantages:**
- More efficient for large datasets
- Window functions for complex aggregations
- Better performance than pandas groupby
- Can handle NULL values elegantly
- SQL-based transformations are reproducible

**Features to Create:**
- user_interaction_rank: Chronological rank of interactions
- user_cumulative_interactions: Running total of interactions per user
- user_interaction_frequency: Interactions per day/week
- user_days_since_first_interaction: Days elapsed since first interaction

In [ ]:
# SQL Query 1: User Interaction Ranking (Chronological)
query_user_ranking = """
SELECT 
    user_id,
    product_id,
    interaction_type,
    timestamp,
    ROW_NUMBER() OVER (PARTITION BY user_id ORDER BY timestamp) as user_interaction_rank,
    COUNT(*) OVER (PARTITION BY user_id) as user_total_interactions,
    SUM(CASE WHEN interaction_type = 'purchase' THEN 1 ELSE 0 END) 
        OVER (PARTITION BY user_id) as user_purchase_count,
    SUM(CASE WHEN interaction_type = 'cart' THEN 1 ELSE 0 END) 
        OVER (PARTITION BY user_id) as user_cart_count,
    SUM(CASE WHEN interaction_type = 'view' THEN 1 ELSE 0 END) 
        OVER (PARTITION BY user_id) as user_view_count
FROM engineered_features
ORDER BY user_id, timestamp
"""

print("\n1. User Interaction Ranking Query")
user_ranking = pd.read_sql_query(query_user_ranking, engine)
print(f"   ✓ Created {len(user_ranking)} records with ranking")
print(f"   Sample:\n{user_ranking[['user_id', 'interaction_type', 'user_interaction_rank', 'user_total_interactions']].head(10)}")

# Merge back to main dataframe
df = df.merge(user_ranking[['user_id', 'product_id', 'user_interaction_rank', 'user_total_interactions']], 
              on=['user_id', 'product_id'], how='left', suffixes=('', '_sql'))

print("\n   ✓ Merged user ranking features to main dataframe")

In [17]:
# 5. Product-level Aggregations
print("Creating product-level aggregations...")
product_aggregations = df.groupby('product_id').agg({
    'user_id': 'nunique',
    'product_rating': 'first',
    'price': 'first',
    'category': 'first',
    'stock': 'first',
    'brand': 'first',
    'has_purchase': 'sum',
    'has_cart': 'sum',
    'product_user_diversity': 'first',
    'product_avg_rating': 'first',
    'product_engagement_level': 'first',
    'product_sales_velocity': 'first'
}).reset_index()

product_aggregations.columns = ['product_id', 'unique_users', 'rating', 'price',
                                'category', 'stock', 'brand', 'purchase_count',
                                'cart_count', 'user_diversity', 'avg_rating',
                                'engagement_level', 'sales_velocity']

product_aggregations.to_sql('product_aggregations', con=engine, if_exists='replace', index=False, chunksize=10000)
del product_aggregations
gc.collect()
print("✓ Product aggregations stored")

Creating product-level aggregations...
✓ Product aggregations stored


## SQL-Based Product Feature Engineering

**Purpose:** Create product-level features using SQL aggregations and rankings

**Queries Include:**
1. Product popularity ranking within category
2. Product price percentile within category
3. Product rating consistency metrics
4. Product conversion funnel stages

**Window Functions Used:**
- ROW_NUMBER(): Rank products by various metrics
- RANK(): Handle ties in rankings
- PERCENT_RANK(): Calculate percentile rankings
- NTILE(): Categorize into buckets (quartiles, deciles)

**Features to Create:**
- product_rank_in_category: Product's position in category by interactions
- product_price_percentile: Price percentile within category
- product_rating_rank: Rating rank within category
- product_conversion_stage: Stage in conversion funnel

In [ ]:
# SQL Query 2: Product Rankings and Percentiles
query_product_ranking = """
SELECT 
    product_id,
    category,
    price,
    product_rating,
    COUNT(*) as product_interaction_count,
    SUM(CASE WHEN interaction_type = 'purchase' THEN 1 ELSE 0 END) as product_purchases,
    RANK() OVER (PARTITION BY category ORDER BY product_rating DESC) as product_rating_rank_in_category,
    RANK() OVER (PARTITION BY category ORDER BY price) as product_price_rank_in_category,
    PERCENT_RANK() OVER (PARTITION BY category ORDER BY price) as product_price_percentile_in_category,
    NTILE(4) OVER (PARTITION BY category ORDER BY product_rating) as product_rating_quartile,
    NTILE(5) OVER (PARTITION BY category ORDER BY COUNT(*)) as product_popularity_quintile
FROM engineered_features
GROUP BY product_id, category, price, product_rating
ORDER BY product_id
"""

print("\n1. Product Ranking Query")
product_ranking = pd.read_sql_query(query_product_ranking, engine)
print(f"   ✓ Created {len(product_ranking)} product ranking records")
print(f"   Sample:\n{product_ranking[['product_id', 'category', 'product_rating_rank_in_category', 'product_price_percentile_in_category']].head(10)}")

# Merge to main dataframe
df = df.merge(product_ranking[['product_id', 'product_rating_rank_in_category', 
                               'product_price_percentile_in_category', 'product_rating_quartile',
                               'product_popularity_quintile']], 
              on='product_id', how='left')

print("\n   ✓ Merged product ranking features to main dataframe")

## SQL-Based User-Product Interaction Features

**Purpose:** Capture detailed interaction history between users and products

**Queries Include:**
1. User's interaction history with specific product
2. Time since last interaction for user-product pair
3. User's previous product interactions (sequential behavior)
4. Repeat purchase indicators

**Features to Create:**
- user_product_interaction_history: Count of previous interactions
- days_since_last_interaction: Recency metric
- user_previous_purchase_with_product: Binary flag
- user_product_sequence_position: Position in sequence

In [ ]:
# SQL Query 3: User-Product Interaction History
query_user_product_history = """
SELECT 
    u1.user_id,
    u1.product_id,
    u1.timestamp,
    COUNT(u2.user_id) as user_product_previous_interactions,
    (julianday(u1.timestamp) - MAX(julianday(u2.timestamp))) as days_since_last_interaction,
    SUM(CASE WHEN u2.interaction_type = 'purchase' THEN 1 ELSE 0 END) as user_product_previous_purchases,
    MAX(CASE WHEN u2.interaction_type = 'view' THEN 1 ELSE 0 END) as user_viewed_product_before,
    MAX(CASE WHEN u2.interaction_type = 'cart' THEN 1 ELSE 0 END) as user_carted_product_before,
    MAX(CASE WHEN u2.interaction_type = 'purchase' THEN 1 ELSE 0 END) as user_purchased_product_before
FROM engineered_features u1
LEFT JOIN engineered_features u2 
    ON u1.user_id = u2.user_id 
    AND u1.product_id = u2.product_id 
    AND u2.timestamp < u1.timestamp
GROUP BY u1.user_id, u1.product_id, u1.timestamp
ORDER BY u1.user_id, u1.product_id, u1.timestamp
"""

print("\n1. User-Product Interaction History Query")
user_product_history = pd.read_sql_query(query_user_product_history, engine)
print(f"   ✓ Created {len(user_product_history)} user-product history records")
print(f"   Sample:\n{user_product_history[['user_id', 'product_id', 'user_product_previous_interactions', 'days_since_last_interaction']].head(10)}")

# Clean up NULL values
user_product_history['days_since_last_interaction'].fillna(999, inplace=True)
user_product_history['user_product_previous_interactions'].fillna(0, inplace=True)

# Merge to main dataframe
df = df.merge(user_product_history[['user_id', 'product_id', 'timestamp', 
                                     'user_product_previous_interactions', 
                                     'days_since_last_interaction',
                                     'user_purchased_product_before']], 
              on=['user_id', 'product_id', 'timestamp'], how='left')

print("\n   ✓ Merged user-product history features to main dataframe")

In [ ]:
# 6. Category-level Aggregations
print("Creating category-level aggregations...")
category_aggregations = df.groupby('category').agg({
    'product_id': 'nunique',
    'price': ['mean', 'std'],
    'product_rating': 'mean',
    'discountPercentage': 'mean',
    'has_purchase': 'sum',
    'category_conversion_rate': 'first'
}).reset_index()

category_aggregations.columns = ['category', 'unique_products', 'avg_price', 'price_std',
                                 'avg_rating', 'avg_discount', 'purchase_count', 'conversion_rate']

category_aggregations.to_sql('category_aggregations', con=engine, if_exists='replace', index=False, chunksize=10000)
del category_aggregations
gc.collect()
print("✓ Category aggregations stored")

print("\n✓ All database operations completed successfully")

Creating category-level aggregations...
✓ Category aggregations stored

✓ All database operations completed successfully


## SQL-Based Category Feature Engineering

**Purpose:** Create category-level aggregations and comparisons

**Queries Include:**
1. Category market statistics
2. Product positioning within category
3. Category trend analysis
4. Competition metrics within category

**Features to Create:**
- category_total_interactions: Total interactions in category
- category_avg_price: Average price in category
- product_price_vs_category_mean: Price deviation from category average
- product_popularity_vs_category: Product's popularity rank
- category_market_concentration: Herfindahl index (concentration)

In [ ]:
# SQL Query 4: Category Market Analysis
query_category_analysis = """
SELECT 
    category,
    COUNT(*) as category_total_interactions,
    COUNT(DISTINCT product_id) as category_unique_products,
    COUNT(DISTINCT user_id) as category_unique_users,
    AVG(price) as category_avg_price,
    MIN(price) as category_min_price,
    MAX(price) as category_max_price,
    AVG(product_rating) as category_avg_rating,
    SUM(CASE WHEN interaction_type = 'purchase' THEN 1 ELSE 0 END) as category_purchases,
    ROUND(CAST(SUM(CASE WHEN interaction_type = 'purchase' THEN 1 ELSE 0 END) AS FLOAT) 
        / COUNT(*) * 100, 2) as category_conversion_rate,
    AVG(discountPercentage) as category_avg_discount
FROM engineered_features
GROUP BY category
ORDER BY category_total_interactions DESC
"""

print("\n1. Category Market Analysis Query")
category_analysis = pd.read_sql_query(query_category_analysis, engine)
print(f"   ✓ Created {len(category_analysis)} category records")
print(f"   Sample:\n{category_analysis[['category', 'category_total_interactions', 'category_avg_price', 'category_conversion_rate']].head(10)}")

# Merge to main dataframe
df = df.merge(category_analysis[['category', 'category_total_interactions', 'category_avg_price',
                                  'category_avg_rating', 'category_conversion_rate']], 
              on='category', how='left')

print("\n   ✓ Merged category features to main dataframe")

## SQL-Based Temporal Feature Engineering

**Purpose:** Create time-based features using SQL date functions

**Queries Include:**
1. Daily/weekly/monthly aggregations
2. Seasonal patterns
3. User activity trends over time
4. Product lifecycle stage

**Features to Create:**
- interaction_hour: Hour of interaction
- interaction_day_of_week: Day of week
- interaction_week_number: Week number
- user_activity_trend: Recent activity vs historical average
- product_lifecycle_stage: Based on interaction trends

In [ ]:
# SQL Query 5: Temporal Features
query_temporal = """
SELECT 
    user_id,
    product_id,
    timestamp,
    strftime('%H', timestamp) as interaction_hour,
    strftime('%w', timestamp) as interaction_day_of_week,
    strftime('%W', timestamp) as interaction_week_number,
    strftime('%m', timestamp) as interaction_month,
    strftime('%Y', timestamp) as interaction_year,
    CASE 
        WHEN CAST(strftime('%H', timestamp) AS INTEGER) BETWEEN 6 AND 11 THEN 'Morning'
        WHEN CAST(strftime('%H', timestamp) AS INTEGER) BETWEEN 12 AND 16 THEN 'Afternoon'
        WHEN CAST(strftime('%H', timestamp) AS INTEGER) BETWEEN 17 AND 20 THEN 'Evening'
        ELSE 'Night'
    END as interaction_period,
    CASE 
        WHEN CAST(strftime('%w', timestamp) AS INTEGER) IN (5, 6) THEN 1
        ELSE 0
    END as is_weekend,
    COUNT(*) OVER (PARTITION BY user_id, DATE(timestamp)) as user_interactions_per_day,
    COUNT(*) OVER (PARTITION BY product_id, DATE(timestamp)) as product_interactions_per_day,
    COUNT(*) OVER (PARTITION BY user_id ORDER BY DATE(timestamp) 
                   ROWS BETWEEN 6 PRECEDING AND CURRENT ROW) as user_interactions_last_7_days
FROM engineered_features
ORDER BY user_id, timestamp
"""

print("\n1. Temporal Features Query")
temporal_features = pd.read_sql_query(query_temporal, engine)
print(f"   ✓ Created {len(temporal_features)} temporal feature records")
print(f"   Sample:\n{temporal_features[['user_id', 'product_id', 'interaction_hour', 'interaction_period', 'user_interactions_last_7_days']].head(10)}")

# Merge to main dataframe
df = df.merge(temporal_features[['user_id', 'product_id', 'timestamp', 'interaction_hour', 
                                  'interaction_day_of_week', 'interaction_period', 'is_weekend',
                                  'user_interactions_last_7_days']], 
              on=['user_id', 'product_id', 'timestamp'], how='left')

print("\n   ✓ Merged temporal features to main dataframe")

## SQL-Based Cohort and Funnel Features

**Purpose:** Create features based on user cohorts and purchase funnel stages

**Queries Include:**
1. User cohort assignment (first interaction date)
2. User lifecycle stage (new, active, churned, returning)
3. Purchase funnel position (view → cart → purchase)
4. User retention metrics

**Features to Create:**
- user_cohort_month: Month when user first interacted
- user_lifecycle_stage: Classification of user stage
- user_funnel_stage: Current position in purchase funnel
- user_days_in_system: Days since first interaction
- user_returning_customer: Flag for repeat customer

In [ ]:
# SQL Query 6: User Cohort Analysis
query_user_cohorts = """
SELECT 
    user_id,
    MIN(DATE(timestamp)) as user_first_interaction_date,
    MAX(DATE(timestamp)) as user_last_interaction_date,
    strftime('%Y-%m', MIN(timestamp)) as user_cohort_month,
    COUNT(DISTINCT DATE(timestamp)) as user_active_days,
    COUNT(*) as user_total_interactions,
    SUM(CASE WHEN interaction_type = 'purchase' THEN 1 ELSE 0 END) as user_total_purchases,
    CASE 
        WHEN SUM(CASE WHEN interaction_type = 'purchase' THEN 1 ELSE 0 END) > 0 THEN 1
        ELSE 0
    END as user_is_customer,
    CASE 
        WHEN COUNT(*) = 1 AND interaction_type = 'view' THEN 'Visitor'
        WHEN SUM(CASE WHEN interaction_type = 'purchase' THEN 1 ELSE 0 END) > 0 
            AND julianday('now') - julianday(MAX(timestamp)) < 30 THEN 'Active Customer'
        WHEN SUM(CASE WHEN interaction_type = 'purchase' THEN 1 ELSE 0 END) > 0 
            AND julianday('now') - julianday(MAX(timestamp)) >= 30 THEN 'Inactive Customer'
        WHEN COUNT(*) > 1 AND SUM(CASE WHEN interaction_type = 'purchase' THEN 1 ELSE 0 END) = 0 THEN 'Browser'
        ELSE 'Unknown'
    END as user_lifecycle_stage
FROM engineered_features
GROUP BY user_id
ORDER BY user_total_interactions DESC
"""

print("\n1. User Cohort Analysis Query")
user_cohorts = pd.read_sql_query(query_user_cohorts, engine)
print(f"   ✓ Created {len(user_cohorts)} user cohort records")
print(f"   Sample:\n{user_cohorts[['user_id', 'user_cohort_month', 'user_lifecycle_stage', 'user_total_purchases']].head(10)}")

# Merge to main dataframe
df = df.merge(user_cohorts[['user_id', 'user_cohort_month', 'user_lifecycle_stage', 'user_first_interaction_date']], 
              on='user_id', how='left')

print("\n   ✓ Merged user cohort features to main dataframe")

## SQL Feature Engineering Summary

**Total SQL Queries Executed: 6**

1. **User Ranking Features** - Chronological interaction ranking
2. **Product Rankings** - Category-based product positioning
3. **User-Product History** - Interaction recency and frequency
4. **Category Analysis** - Market-level statistics
5. **Temporal Features** - Time-based categorization
6. **Cohort Analysis** - User lifecycle and segments

**New Features Created via SQL: 25+**

**Benefits of SQL Approach:**
- ✓ Efficient aggregations at scale
- ✓ Window functions for complex logic
- ✓ Reproducible transformations
- ✓ Better performance on large datasets
- ✓ SQL-based auditing and validation

## Feature Scaling and Normalization

**Purpose:** Prepare features for machine learning models

**Scaling Methods:**

1. **StandardScaler (Z-score):**
   - Transform: (X - mean) / std
   - Use for: Linear models, Neural Networks
   - Result: Mean ≈ 0, Std ≈ 1

2. **MinMaxScaler:**
   - Transform: (X - min) / (max - min)
   - Use for: Tree-based models, Distance-based algorithms
   - Result: Range [0, 1]

3. **RobustScaler:**
   - Transform: (X - median) / IQR
   - Use for: Data with outliers
   - Result: Robust to extreme values

**Features to Scale:**
- All numerical features (ratios, counts, prices)
- Keep categorical features as-is
- Preserve engineered features

In [19]:
from sklearn.preprocessing import StandardScaler, MinMaxScaler, RobustScaler

# Identify numerical features for scaling
numerical_features = df.select_dtypes(include=[np.number]).columns.tolist()

print(f"\nNumerical features to scale: {len(numerical_features)}")
print(f"  {numerical_features[:10]}... and {len(numerical_features) - 10} more")

# Create scaled versions
scaler_standard = StandardScaler()
scaler_minmax = MinMaxScaler()
scaler_robust = RobustScaler()

# Apply standard scaling
df_scaled = df.copy()
scaled_cols_standard = [f"{col}_scaled" for col in numerical_features]
df_scaled[scaled_cols_standard] = scaler_standard.fit_transform(df[numerical_features])

# Apply MinMax scaling
scaled_cols_minmax = [f"{col}_minmax" for col in numerical_features]
df_scaled[scaled_cols_minmax] = scaler_minmax.fit_transform(df[numerical_features])

# Apply Robust scaling
scaled_cols_robust = [f"{col}_robust" for col in numerical_features]
df_scaled[scaled_cols_robust] = scaler_robust.fit_transform(df[numerical_features])

print("\n✓ Features scaled using three methods")
print(f"  - StandardScaler: {len(scaled_cols_standard)} features")
print(f"  - MinMaxScaler: {len(scaled_cols_minmax)} features")
print(f"  - RobustScaler: {len(scaled_cols_robust)} features")

print(f"\nScaled feature examples (first 5 records):")
print(df_scaled[[numerical_features[0], scaled_cols_standard[0], 
                 scaled_cols_minmax[0], scaled_cols_robust[0]]].head())


Numerical features to scale: 82
  ['user_id', 'product_id', 'user_rating', 'id', 'price', 'discountPercentage', 'product_rating', 'stock', 'weight', 'minimumOrderQuantity']... and 72 more

✓ Features scaled using three methods
  - StandardScaler: 82 features
  - MinMaxScaler: 82 features
  - RobustScaler: 82 features

Scaled feature examples (first 5 records):
   user_id  user_id_scaled  user_id_minmax  user_id_robust
0      694       -1.459600        0.069307       -0.840047
1     2276       -0.915155        0.227523       -0.527646
2     4642       -0.100895        0.464146       -0.060427
3     3016       -0.660483        0.301530       -0.381517
4     8246        1.139423        0.824582        0.651264


## Feature Engineering Complete

**Summary of Created Features:**

**Interaction Features (7 features):**
- user_product_interaction_count
- user_product_diversity
- product_user_diversity
- interaction_type_encoded
- has_purchase, has_cart, has_view

**User Behavior Features (7 features):**
- user_avg_rating, user_rating_std
- user_avg_price, user_price_std
- user_preferred_category
- user_preferred_price_range
- user_repeat_purchase_rate

**Product Performance Features (8 features):**
- product_avg_rating, product_rating_std
- product_rating_gap
- product_rating_tier
- product_sales_velocity
- product_view_to_purchase_ratio
- product_discount_frequency
- product_stock_level

**Category Features (9 features):**
- category_avg_price, category_price_std
- category_avg_rating
- category_conversion_rate
- product_price_vs_category
- price_premium_ratio
- product_rating_vs_category

**Derived Features (9 features):**
- price_quality_ratio
- discount_effectiveness
- cart_to_view_ratio
- purchase_to_cart_ratio
- high_rating_budget_product
- discount_vulnerable_product
- user_category_match

**Temporal Features (from EDA):**
- hour, day_of_week, day_period
- is_weekend, month, year

**Total New Features Created: 50+**

**Ready for Model Building!**

In [21]:
print(f"\nFinal Dataset Statistics:")
print(f"  Total records: {len(df):,}")
print(f"  Total features: {len(df.columns)}")
print(f"  Memory usage: {df.memory_usage(deep=True).sum() / (1024**2):.2f} MB")

print(f"\nFeature Distribution:")
numerical_count = len(df.select_dtypes(include=[np.number]).columns)
categorical_count = len(df.select_dtypes(include=['object', 'category']).columns)
boolean_count = len(df.select_dtypes(include=['bool']).columns)

print(f"  - Numerical: {numerical_count}")
print(f"  - Categorical: {categorical_count}")
print(f"  - Boolean: {boolean_count}")

print(f"\nEngineered Features Created:")
engineered_features = ['interaction_type_encoded', 'has_purchase', 'has_cart',
                       'user_engagement_level', 'product_engagement_level',
                       'price_quality_ratio', 'discount_effectiveness',
                       'cart_to_view_ratio', 'purchase_to_cart_ratio',
                       'high_rating_budget_product', 'user_category_match']

for i, feature in enumerate(engineered_features, 1):
    if feature in df.columns:
        print(f"  {i}. ✓ {feature}")

print(f"\nData Quality Checks:")
print(f"  - Missing values: {df.isnull().sum().sum()}")
print(f"  - Duplicate records: {df.duplicated().sum()}")
print(f"  - Infinite values: {np.isinf(df.select_dtypes(include=[np.number])).sum().sum()}")



Final Dataset Statistics:
  Total records: 26,250
  Total features: 110
  Memory usage: 70.29 MB

Feature Distribution:
  - Numerical: 82
  - Categorical: 27
  - Boolean: 1

Engineered Features Created:
  1. ✓ interaction_type_encoded
  2. ✓ has_purchase
  3. ✓ has_cart
  4. ✓ user_engagement_level
  5. ✓ product_engagement_level
  6. ✓ price_quality_ratio
  7. ✓ discount_effectiveness
  8. ✓ cart_to_view_ratio
  9. ✓ purchase_to_cart_ratio
  10. ✓ high_rating_budget_product
  11. ✓ user_category_match

Data Quality Checks:
  - Missing values: 79614
  - Duplicate records: 16254
  - Infinite values: 0


In [22]:
print(f"\nExporting final engineered dataset...")
output_file = r'e:/dm4ml project/data/processed/final_dataset_transformed.csv'
df_scaled.to_csv(output_file, index=False)
print(f"  ✓ Saved to: {output_file}")


Exporting final engineered dataset...
  ✓ Saved to: e:/dm4ml project/data/processed/final_dataset_transformed.csv
